# Do Mainstream Young Singles/Couples Pay More per Pack of Chips?

## Question
Do loyalty-card customers in the Mainstream, Young Singles/Couples segment
pay a higher average price per pack of chips than all other customers, and
is the difference larger than chance?

## Data source
Quantium retail analytics simulation, two files:
- `Q_Transaction_Data` : chip purchases, 1 July 2018 to 30 June 2019
- `Q_Purchase_Behaviour` : loyalty-card customer segments

## Unit of observation
- Raw data: one product line within a purchase.
- For the statistical test: one loyalty card.

## Important variables
- `LYLTY_CARD_NBR` : loyalty card, used to join the two files
- `LIFESTAGE`, `PREMIUM_CUSTOMER` : customer segment labels
- `PROD_QTY`, `TOT_SALES` : packs bought and dollars spent
- `price_per_pack` = `TOT_SALES / PROD_QTY` (derived)

## Main data limitations
- One year of data, chips only, loyalty-card customers only.
- Segment definitions are not documented by the source.
- The data can show association, not cause.

In [1]:
import pandas as pd

import sys
from pathlib import Path
sys.path.append(str(Path("../src").resolve()))

from cleaning import (
    standardize_product_names,
    remove_non_chip_products,
    remove_outliers,
)

from features import (
    parse_pack_size,
    parse_brand,
    add_price_per_pack,
)

In [2]:
## LOADING DATA

trans = pd.read_csv("../data/raw/Q_Transaction_Data.csv", parse_dates=["DATE"])
cust = pd.read_csv("../data/raw/Q_Purchase_Behaviour.csv")

CHECKING FOR PROBLEMS AND CLEANING

In [3]:
# Problem 1: exact duplicate rows
print("Duplicate rows:", trans.duplicated().sum())
trans.loc[trans.duplicated(keep=False)]

Duplicate rows: 1


,DATE,STORE_NBR,LYLTY_CARD_NBR,TXN_ID,PROD_NBR,PROD_NAME,PROD_QTY,TOT_SALES
124843,2018-10-01,107,107024,108462,45,Smiths Thinly Cut Roast Chicken 175g,2,6.0
124845,2018-10-01,107,107024,108462,45,Smiths Thinly Cut Roast Chicken 175g,2,6.0


In [4]:
# Problem 2: extreme quantities
trans.loc[trans["PROD_QTY"] > 10]

,DATE,STORE_NBR,LYLTY_CARD_NBR,TXN_ID,PROD_NBR,PROD_NAME,PROD_QTY,TOT_SALES
69762,2018-08-19,226,226000,226201,4,Dorito Corn Chp Supreme 380g,200,650.0
69763,2019-05-20,226,226000,226210,4,Dorito Corn Chp Supreme 380g,200,650.0


In [5]:
# Is that card normally a big buyer?
trans.loc[trans["LYLTY_CARD_NBR"] == 226000]

,DATE,STORE_NBR,LYLTY_CARD_NBR,TXN_ID,PROD_NBR,PROD_NAME,PROD_QTY,TOT_SALES
69762,2018-08-19,226,226000,226201,4,Dorito Corn Chp Supreme 380g,200,650.0
69763,2019-05-20,226,226000,226210,4,Dorito Corn Chp Supreme 380g,200,650.0


In [6]:
# Problem 3: products that are not chips
names = trans["PROD_NAME"].str.replace(r"\s+", " ", regex=True).str.strip()
names.loc[names.str.contains("salsa", case=False)].unique()[:10]

<StringArray>
['Old El Paso Salsa Dip Tomato Mild 300g',
  'Red Rock Deli SR Salsa & Mzzrlla 150g',
   'Smiths Crinkle Cut Tomato Salsa 150g',
              'Doritos Salsa Medium 300g',
 'Old El Paso Salsa Dip Chnky Tom Ht300g',
             'Woolworths Mild Salsa 300g',
  'Old El Paso Salsa Dip Tomato Med 300g',
           'Woolworths Medium Salsa 300g',
                'Doritos Salsa Mild 300g']
Length: 9, dtype: str

In [7]:
log = {"raw rows": len(trans)}

clean = trans.copy()
clean["PROD_NAME"] = standardize_product_names(clean["PROD_NAME"])

n_before = len(clean)
clean = clean.drop_duplicates()
log["exact duplicates removed"] = n_before - len(clean)

clean, salsa = remove_non_chip_products(clean, "salsa")
log["salsa lines removed"] = len(salsa)

clean, bulk = remove_outliers(clean, "PROD_QTY", max_qty=10)
log["bulk-purchase lines removed"] = len(bulk)

log["rows kept"] = len(clean)
pd.Series(log, name="count").to_frame()

,count
raw rows,264836
exact duplicates removed,1
salsa lines removed,18094
bulk-purchase lines removed,2
rows kept,246739


Salsa was removed because the question is about chips. Two bulk purchases of
200 packs from one loyalty card (card 226000) were removed as likely
commercial buying, since that card made no other purchases.

In [8]:
print("Nulls in transactions:", int(clean.isna().sum().sum()))
print("Nulls in customers:", int(cust.isna().sum().sum()))

all_days = pd.date_range(clean["DATE"].min(), clean["DATE"].max())
print(all_days.difference(pd.DatetimeIndex(clean["DATE"].unique())))

Nulls in transactions: 0
Nulls in customers: 0
DatetimeIndex(['2018-12-25'], dtype='datetime64[us]', freq='D')


**Missingness.** Neither file has any empty cells, so no values needed to be
filled in or dropped. One calendar day, 25 December 2018, has no transactions.
This is consistent with stores being closed on Christmas Day, so it is treated
as a real-world gap and not a data error. No imputation was needed.

Features And the Join

In [9]:
clean["PROD_NAME"].sample(8, random_state=1).tolist()

['Thins Chips Seasonedchicken 175g',
 'Dorito Corn Chp Supreme 380g',
 'RRD Lime & Pepper 165g',
 'Thins Chips Seasonedchicken 175g',
 'Cobs Popd Sour Crm &Chives Chips 110g',
 'Infuzions Thai SweetChili PotatoMix 110g',
 'Tyrrells Crisps Lightly Salted 165g',
 'Kettle 135g Swt Pot Sea Salt']

In [10]:
clean["pack_g"] = parse_pack_size(clean["PROD_NAME"])
clean["brand"] = parse_brand(clean["PROD_NAME"])
clean = add_price_per_pack(clean)

print("Missing pack sizes:", clean["pack_g"].isna().sum())
clean["brand"].value_counts().head(15)

Missing pack sizes: 0


brand
KETTLE       41288
SMITHS       30352
DORITOS      25224
PRINGLES     25102
RRD          16321
INFUZIONS    14201
THINS        14075
WW           11836
COBS          9693
TOSTITOS      9471
TWISTIES      9454
GRAIN         7740
NATURAL       7469
TYRRELLS      6442
CHEEZELS      4603
Name: count, dtype: int64

**Pack size and brand.** I extracted pack size from the product name by identifying the number immediately before "g" (for example, `175g`). All products had an identifiable pack size. I identified the brand using the first word of the product name.

However, some brands appeared under different spellings or abbreviations, such as `SMITH` and `SMITHS`, `RED` and `RRD`, and `DORITO` and `DORITOS`. To avoid splitting sales belonging to the same brand across multiple labels, I standardised eight variants into their corresponding brand names using the `BRAND_FIXES` mapping in `src/features.py`. This reduced the number of distinct brand names from 28 to 20.

**Limitation:** Since brand identification relies on the first word of the product name, some labels, such as `BURGER` and `FRENCH`, represent product lines rather than actual brands.

In [11]:
# The customer table must have one row per card
print("Cards unique in customer table:", cust["LYLTY_CARD_NBR"].is_unique)

Cards unique in customer table: True


In [12]:
# Diagnose the join before trusting it
check = clean.merge(
    cust, on="LYLTY_CARD_NBR", how="left", indicator=True
)
print(check["_merge"].value_counts())

_merge
both          246739
left_only          0
right_only         0
Name: count, dtype: int64


In [13]:
# The real join, with a built-in relationship check
data = clean.merge(
    cust, on="LYLTY_CARD_NBR", how="left", validate="many_to_one"
)

print("Rows:", len(data))
print("Missing lifestage:", data["LIFESTAGE"].isna().sum())
print("Brands:", data["brand"].nunique())

Rows: 246739
Missing lifestage: 0
Brands: 20


**Joining transactions to customers.** The transaction file contains details of each purchase but does not identify the customer, while the customer file links each loyalty card to its owner but does not contain purchase information. To compare customer segments, I joined the two datasets using `LYLTY_CARD_NBR`.

This is a many-to-one relationship because a customer can have multiple purchase transactions, but each loyalty card appears only once in the customer table. I used a left join to retain all purchase transactions and `validate="many_to_one"` to ensure that each transaction matched no more than one customer record.

**Validation checks:** All 246,739 purchase transactions had a matching customer record, as confirmed by the absence of `left_only` rows in the merge indicator. The row count remained unchanged at 246,739 after the join, and there were no missing values in the lifestage column.

In [14]:
data.to_csv("../data/processed/processed_data.csv", index=False)